In [ ]:
import csv, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # repo root on path
import cleaned_data as cd
from cleaned_data import cleaning_utils as cu, db, embeddings, clustering
csv.field_size_limit(10**9)


In [ ]:
rows = []
with open(cd.SCORECARD_CSV, encoding="utf-8-sig", newline="") as f:
    for r in csv.DictReader(f):
        if not cu.is_real_call(r):
            continue
        band, raw = cu.normalize_grade(r.get("grade", ""))
        name, email, slug = cu.canonicalize_rep(r.get("rep_name", ""),
                                                r.get("Rep Email", ""))
        rows.append({"row": r, "band": band, "raw": raw, "name": name,
                     "email": email, "slug": slug,
                     "has_num": cu.has_numeric_score(r)})
print(f"kept {len(rows)} real calls; "
      f"narrative-only (no numeric score): {sum(1 for x in rows if not x['has_num'])}")


In [ ]:
conn = db.connect(cd.DB_PATH)
db.create_schema(conn)
for x in rows:
    r = x["row"]
    rid = db.upsert_rep(conn, x["name"], x["email"], x["slug"])
    ts = (r.get("total_score") or "").strip()
    db.insert_call(conn, rid, {
        "client_name": r.get("client_name"), "call_date": r.get("call_date"),
        "show_name": r.get("show_name"), "meeting_id": r.get("meeting_id"),
        "total_score": float(ts) if ts else None,
        "grade_normalized": x["band"], "grade_raw": x["raw"],
        "close_ask": {True: 1, False: 0, None: None}[
            cu.parse_close_ask(r.get("did_rep_ask_for_close", ""))],
        "has_numeric_score": 1 if x["has_num"] else 0,
        "intended_outcome": r.get("intended_outcome"),
        "deal_outcome_context": r.get("deal_outcome_context"),
        "flagged_followup": r.get("Flagged For Follow-Up (AI)"),
        "one_line_verdict": r.get("one_line_verdict"),
        "biggest_strength": r.get("biggest_strength"),
        "what_went_well": r.get("what_went_well"),
        "what_made_close_work": r.get("what_made_this_close_work"),
        "what_to_improve": r.get("what_to_improve"),
        "why_no_close": r.get("why_no_close"), "red_flags": r.get("red_flags"),
        "coaching_tip": r.get("coaching_tip"),
        "rep_improvement": r.get("Rep Improvement Suggestions (AI)"),
        "rudys_note": r.get("rudys_note"),
        "objections_surfaced": r.get("objections_surfaced")})
print("reps:", conn.execute("SELECT COUNT(*) FROM reps").fetchone()[0],
      "calls:", conn.execute("SELECT COUNT(*) FROM calls").fetchone()[0])


**Requires `OPENROUTER_API_KEY`.** The cell below calls OpenRouter (via `cleaned_data.embeddings.make_client`) to embed phrases and an LLM to label clusters. It will raise `KeyError` if the key is not set in the environment / `.env`. Do not run it without a valid key — it produces the draft taxonomy YAMLs (`objection_types.yaml`, `weakness_types.yaml`) that are reviewed by hand in Stage 2 (Task 15) before anything downstream trusts them.


In [ ]:
import yaml
client = embeddings.make_client()

def propose(phrases, kind):
    vecs = embeddings.embed_texts(phrases, client)
    labels = embeddings.cluster_vectors(vecs, min_cluster_size=8)
    groups = embeddings.group_by_cluster(phrases, labels)
    out = []
    for i, (_, members) in enumerate(sorted(groups.items()), start=1):
        lab = clustering.label_cluster(members, client)
        out.append({"id": i, **lab, "n_examples": len(members),
                    "example_quotes": members[:3]})
    cd.TAXONOMY_DIR.mkdir(parents=True, exist_ok=True)
    (cd.TAXONOMY_DIR / f"{kind}.yaml").write_text(
        yaml.safe_dump(out, sort_keys=False, allow_unicode=True), encoding="utf-8")
    print(f"{kind}: {len(out)} clusters from {len(phrases)} phrases")
    return out

obj_phrases, weak_phrases = [], []
for c in conn.execute("SELECT objections_surfaced, what_to_improve, why_no_close, "
                      "red_flags FROM calls"):
    obj_phrases += cu.extract_objection_phrases(c["objections_surfaced"] or "")
    blob = cu.pool_weakness_text(dict(c))
    if blob:
        weak_phrases.append(blob)

propose(obj_phrases, "objection_types")
propose(weak_phrases, "weakness_types")
